# Model vs recency-vote rule

Stage 3 trains a shallow histogram-gradient booster on the feature table.
Validation is **grouped by person**: a candidate's T0 row never trains a model
that is then tested on that same candidate's T1 row.

The number to quote is nested-CV / OOF performance, not the fit-on-all-300
score. Details are in `docs/DATA_FLOW.md`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "oos_review").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from oos_review.evaluate import summarize
from oos_review.paths import MODEL_DIR, BASELINE_DIR, PROJECT_ROOT as ROOT

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)

cv = pd.read_csv(MODEL_DIR / "cv_fold_metrics.csv")
oof = pd.read_csv(MODEL_DIR / "oof_predictions.csv")
audit = pd.read_csv(MODEL_DIR / "case_predictions_audit.csv")
baseline = pd.read_csv(BASELINE_DIR / "case_predictions_audit.csv")
submission = pd.read_csv(ROOT / "case_predictions.csv")
print("CV folds")
print(cv.to_string(index=False))
print()
print(
    "outer-fold mean  "
    f"acc={cv.accuracy.mean():.3f}  "
    f"macro-F1={cv.macro_f1.mean():.3f}  "
    f"log-loss={cv.log_loss.mean():.3f}"
)


CV folds
 fold  accuracy  macro_f1  log_loss  n_people                                                                                                        params
    0  0.566667  0.561954  1.122419        60 {'clf__l2_regularization': 0.5, 'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__min_samples_leaf': 15}
    1  0.458333  0.446511  1.176996        60 {'clf__l2_regularization': 0.5, 'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__min_samples_leaf': 25}
    2  0.450000  0.451193  1.359737        60 {'clf__l2_regularization': 1.0, 'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__min_samples_leaf': 25}
    3  0.491667  0.486755  1.172567        60 {'clf__l2_regularization': 0.5, 'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__min_samples_leaf': 25}
    4  0.391667  0.398818  1.159072        60 {'clf__l2_regularization': 1.0, 'clf__learning_rate': 0.08, 'clf__max_depth': 2, 'clf__min_samples_leaf': 15}

outer-fold mean  acc=0.472  macro-F1=0.469  log-loss=1

## OOF vs the rule

Both compared on the same 600 labeled rows (300 people × T0/T1). The rule
does not train, so its labeled accuracy is already an honest number. The
model's honest number is OOF, not in-sample.


In [2]:
rule_labeled = baseline.merge(
    oof[["candidate_record_id", "phase", "y"]],
    on=["candidate_record_id", "phase"],
)
print("Rule on labeled rows")
sr = summarize(rule_labeled["y"], rule_labeled["predicted_class"])
print(f"acc={sr['accuracy']:.3f}  macro-F1={sr['macro_f1']:.3f}")
print(sr["per_class"].round(3).to_string(index=False))
print()
print("Model OOF")
sm = summarize(oof["y"], oof["oof_predicted_class"])
print(f"acc={sm['accuracy']:.3f}  macro-F1={sm['macro_f1']:.3f}")
print(sm["per_class"].round(3).to_string(index=False))
print(sm["confusion"].to_string())
print()
for phase in ["T0", "T1"]:
    sub = oof.loc[oof.phase.eq(phase)]
    print(f"{phase} OOF acc={(sub.y == sub.oof_predicted_class).mean():.3f}")


Rule on labeled rows
acc=0.462  macro-F1=0.463
                class  precision  recall    f1  support
     review_warranted      0.473   0.531 0.500      179
 review_not_warranted      0.549   0.498 0.522      213
insufficient_evidence      0.369   0.365 0.367      208

Model OOF
acc=0.472  macro-F1=0.468
                class  precision  recall    f1  support
     review_warranted      0.480   0.464 0.472      179
 review_not_warranted      0.543   0.592 0.566      213
insufficient_evidence      0.379   0.356 0.367      208
pred                   review_warranted  review_not_warranted  insufficient_evidence
true                                                                                
review_warranted                     83                    32                     64
review_not_warranted                 30                   126                     57
insufficient_evidence                60                    74                     74

T0 OOF acc=0.497
T1 OOF acc=0.447


## Does T1 still move the call?

If T0 and T1 predictions were identical, later evidence would be ignored.


In [3]:
wide = audit.pivot(index="candidate_record_id", columns="phase", values="predicted_class")
print("share of cases whose class changes T0→T1:", float((wide["T0"] != wide["T1"]).mean()))
print("model agrees with rule (all 24,000 rows):", float(audit["model_agrees_with_rule"].mean()))
print()
print("T0 mix:", audit.loc[audit.phase.eq("T0"), "predicted_class"].value_counts().to_dict())
print("T1 mix:", audit.loc[audit.phase.eq("T1"), "predicted_class"].value_counts().to_dict())
print()
print("submission rows", len(submission))
prob_sum = (
    submission.p_review_warranted
    + submission.p_review_not_warranted
    + submission.p_insufficient_evidence
)
print("prob sums to 1:", bool((prob_sum - 1).abs().max() < 1e-6))


share of cases whose class changes T0→T1: 0.2881666666666667
model agrees with rule (all 24,000 rows): 0.611625

T0 mix: {'review_not_warranted': 4441, 'review_warranted': 3986, 'insufficient_evidence': 3573}
T1 mix: {'review_not_warranted': 4087, 'insufficient_evidence': 4029, 'review_warranted': 3884}

submission rows 24000
prob sums to 1: True


## Where the model disagrees with the rule

Disagreements are the cases a reviewer should look at first when checking
whether the booster is doing something other than copying `de_oos_score`.


In [4]:
disagree = audit.loc[~audit.model_agrees_with_rule].copy()
print("disagreement mix (model vs rule)")
print(
    pd.crosstab(
        disagree["rule_predicted_class"],
        disagree["predicted_class"],
        margins=True,
    ).to_string()
)
print()
cols = [
    "candidate_record_id", "phase", "de_oos_score",
    "predicted_class", "rule_predicted_class",
    "p_review_warranted", "review_priority", "model_reason",
]
print(disagree.sort_values("model_margin").head(8)[cols].to_string(index=False))


disagreement mix (model vs rule)
predicted_class        insufficient_evidence  review_not_warranted  review_warranted   All
rule_predicted_class                                                                      
insufficient_evidence                      0                  1930              2180  4110
review_not_warranted                    1797                     0               351  2148
review_warranted                        2497                   566                 0  3063
All                                     4294                  2496              2531  9321

candidate_record_id phase  de_oos_score       predicted_class rule_predicted_class  p_review_warranted  review_priority                                                              model_reason
     CAN-N99N7DJ2B8    T1          -2.0 insufficient_evidence review_not_warranted            0.276389         0.297743 model insufficient_evidence (p=0.28/0.36/0.36); rule review_not_warranted
     CAN-950G1340VO    T1       

## Linear companion (not submitted)

`open_address_is_de` and `de_oos_score` have the expected signs. Sparse
state dummies in this table are not reliable with 300 people.


In [5]:
coef = pd.read_csv(MODEL_DIR / "logistic_coefficients.csv").set_index("feature")
core = [
    "num__de_oos_score",
    "num__open_address_is_de",
    "num__title_recency_vote",
    "num__oos_observed_open_de",
    "num__n_t1",
]
print(coef.loc[[c for c in core if c in coef.index]].round(3).to_string())


                           insufficient_evidence  review_not_warranted  review_warranted
feature                                                                                 
num__de_oos_score                         -0.055                -0.124             0.179
num__open_address_is_de                   -0.009                -0.386             0.395
num__title_recency_vote                   -0.163                -0.042             0.205
num__oos_observed_open_de                 -0.032                -0.043             0.075
num__n_t1                                  0.086                -0.056            -0.030


## Submit

The file to send is `case_predictions.csv` at the package root. The rule
file remains in `outputs/baseline/` as the auditable baseline.
